# Tema: Schema enforcement y evolution

## Objetivos
Detectar una columna inesperada y habilitar evolución de forma localizada.

## Conceptos importantes para el examen
Contrato de esquema; mergeSchema en escritura; ALTER ADD COLUMNS; evolución de esquema no sustituye validación de valores.

**Dificultad:** Intermedio · **Tiempo estimado:** 55 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_08_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
employees = spark.createDataFrame(
    [(i, f"Empleado {i:02d}", ["Data", "Sales", "Finance"][i % 3],
      30000 + i * 1500, i % 4 != 0,
      datetime(2026, 1, 1), datetime(2026, 1, 1)) for i in range(1, 19)],
    "employee_id INT, name STRING, department STRING, salary INT, active BOOLEAN, created_at TIMESTAMP, updated_at TIMESTAMP"
)
employees.createOrReplaceTempView("employees_seed")
employees.write.format("delta").mode("overwrite").saveAsTable("employees")
display(employees.orderBy("employee_id"))

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Destino con esquema conocido

In [ ]:
spark.sql("CREATE TABLE schema_demo USING DELTA AS SELECT employee_id, name FROM employees")
extra_column = employees.filter("employee_id <= 3").select("employee_id", "name").withColumn("country", F.lit("ES"))
display(extra_column)

### 2. Fallo esperado controlado
No se captura cualquier error como éxito: comprueba el mensaje. Si falla por permisos o cómputo, corrige esa causa.

In [ ]:
try:
    extra_column.write.format("delta").mode("append").saveAsTable("schema_demo")
except Exception as exc:
    message = str(exc)
    if not any(token in message.lower() for token in ["schema mismatch", "schema_mismatch", "_legacy_error_temp_delta_0007", "delta_merge_incompatible"]):
        raise
    print("Rechazo de esquema observado:", message[:800])
else:
    raise AssertionError("Se esperaba enforcement; revisa si el entorno activa evolución global")

### 3. Evolución explícita

In [ ]:
extra_column.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("schema_demo")
spark.table("schema_demo").printSchema()
display(spark.table("schema_demo"))

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Crea schema_practice con employee_id, name y salary.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Inserta empleado 99 con nueva columna country usando evolución local.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Verifica que las 18 filas originales tienen country NULL y que la nueva contiene ES.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Añade email STRING mediante DDL y actualiza solo el 99.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Genera dos salarios de entrada como texto ('45000' y 'error'), conviértelos y separa cuarentena antes de escribir.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** CTAS y proyección.

**Pista 2:** DataFrameWriter.option.

**Pista 3:** La evolución no rellena retrospectivamente valores.

**Pista 4:** ALTER TABLE ADD COLUMNS.

**Pista 5:** try_cast y filtro; no fuerces un cambio de INT a STRING.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
%sql
CREATE OR REPLACE TABLE schema_practice USING DELTA AS SELECT employee_id, name, salary FROM employees;

### Solución 2

In [ ]:
incoming = spark.createDataFrame([(99, "Nueva", 42000, "ES")], "employee_id INT, name STRING, salary INT, country STRING")
incoming.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("schema_practice")

### Solución 3

In [ ]:
assert spark.table("schema_practice").filter("country IS NULL").count() == 18
assert spark.table("schema_practice").filter("employee_id = 99").first().country == "ES"

### Solución 4

In [ ]:
%sql
ALTER TABLE schema_practice ADD COLUMNS (email STRING);
UPDATE schema_practice SET email = 'persona99@example.invalid' WHERE employee_id = 99;
DESCRIBE TABLE schema_practice;

### Solución 5

In [ ]:
incoming_text = spark.createDataFrame([(100, "45000"), (101, "error")], "employee_id INT, raw_salary STRING")
parsed = incoming_text.withColumn("salary", F.expr("try_cast(raw_salary AS INT)"))
display(parsed.filter("salary IS NULL"))
parsed.filter("salary IS NOT NULL").select("employee_id", F.lit("Alta").alias("name"), "salary", F.lit("ES").alias("country"), F.lit(None).cast("string").alias("email")).write.format("delta").mode("append").saveAsTable("schema_practice")

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
¿Qué activa evolución para una sola escritura append?

A. overwriteSchema sin overwrite

B. mergeSchema=true en esa escritura

C. DROP TABLE

D. VACUUM

### Pregunta 2
¿Qué tienen las filas anteriores en una columna añadida?

A. Siempre cero

B. Un valor aleatorio

C. NULL salvo relleno posterior

D. La cadena 'NULL'

### Pregunta 3
¿Qué evita aceptar 'error' como salario válido?

A. Validación de valor y cuarentena

B. Añadir cualquier columna

C. OPTIMIZE

D. Cambiar de catálogo

### Respuestas y explicación
**1. B** — Se limita el cambio a esa operación.

**2. C** — No existe un valor histórico para esa columna.

**3. A** — La evolución del esquema no resuelve calidad del contenido.

## PARTE 6 - RETO FINAL
Acepta una nueva columna opcional region, rechaza salarios inválidos y demuestra que el esquema se amplía sin convertir salary en texto.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
